<p style="text-align:center">
    <a href="https://skills.network/?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMDS0321ENSkillsNetwork26802033-2022-01-01" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


# **Hands-on Lab: Interactive Visual Analytics with Folium**


Estimated time needed: **40** minutes


The launch success rate may depend on many factors such as payload mass, orbit type, and so on. It may also depend on the location and proximities of a launch site, i.e., the initial position of rocket trajectories. Finding an optimal location for building a launch site certainly involves many factors and hopefully we could discover some of the factors by analyzing the existing launch site locations.


In the previous exploratory data analysis labs, you have visualized the SpaceX launch dataset using `matplotlib` and `seaborn` and discovered some preliminary correlations between the launch site and success rates. In this lab, you will be performing more interactive visual analytics using `Folium`.


## Objectives


This lab contains the following tasks:

*   **TASK 1:** Mark all launch sites on a map
*   **TASK 2:** Mark the success/failed launches for each site on the map
*   **TASK 3:** Calculate the distances between a launch site to its proximities

After completed the above tasks, you should be able to find some geographical patterns about launch sites.


Let's first import required Python packages for this lab:


In [1]:
import piplite
await piplite.install(['folium'])
await piplite.install(['pandas'])

In [2]:
import folium
import pandas as pd

In [3]:
# Import folium MarkerCluster plugin
from folium.plugins import MarkerCluster
# Import folium MousePosition plugin
from folium.plugins import MousePosition
# Import folium DivIcon plugin
from folium.features import DivIcon

If you need to refresh your memory about folium, you may download and refer to this previous folium lab:


[Generating Maps with Python](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/labs/v4/DV0101EN-Exercise-Generating-Maps-in-Python.ipynb)


In [4]:
# Task 1: Mark all launch sites on a map
# Import necessary libraries
import folium
import pandas as pd
from folium.plugins import MarkerCluster
from js import fetch
import io

# 1. Load the data from the URL
URL = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv'
resp = await fetch(URL)
spacex_csv_file = io.BytesIO((await resp.arrayBuffer()).to_py())
spacex_df = pd.read_csv(spacex_csv_file)

# 2. Select and group the data to get unique launch sites and their coordinates
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()
launch_sites_df = launch_sites_df[['Launch Site', 'Lat', 'Long']]

# 3. Create a map and a MarkerCluster object
# Calculate the average lat/long to center the map
avg_lat = launch_sites_df['Lat'].mean()
avg_long = launch_sites_df['Long'].mean()
site_map = folium.Map(location=[avg_lat, avg_long], zoom_start=4)

# Initialize a MarkerCluster object
marker_cluster = MarkerCluster().add_to(site_map)

# 4. Loop through each row of the launch_sites_df DataFrame
for index, record in launch_sites_df.iterrows():
    # Get the coordinates and launch site name
    coordinate = [record['Lat'], record['Long']]
    site_name = record['Launch Site']

    # Create and add a Circle for each launch site
    folium.Circle(
        coordinate,
        radius=1000,
        color='#34A853',
        fill=True,
        fill_color='#34A853'
    ).add_child(folium.Popup(site_name)).add_to(site_map)

    # Create and add a Marker to the MarkerCluster
    folium.Marker(
        coordinate,
        popup=site_name,
        icon=folium.Icon(color='blue')
    ).add_to(marker_cluster)

# 5. Display the map
site_map

First, let's try to add each site's location on a map using site's latitude and longitude coordinates


The following dataset with the name `spacex_launch_geo.csv` is an augmented dataset with latitude and longitude added for each site.


In [5]:
# Download and read the `spacex_launch_geo.csv`
from js import fetch
import io

URL = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv'
resp = await fetch(URL)
spacex_csv_file = io.BytesIO((await resp.arrayBuffer()).to_py())
spacex_df=pd.read_csv(spacex_csv_file)

Now, you can take a look at what are the coordinates for each site.


In [6]:
# Select relevant sub-columns: `Launch Site`, `Lat(Latitude)`, `Long(Longitude)`, `class`
spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()
launch_sites_df = launch_sites_df[['Launch Site', 'Lat', 'Long']]
launch_sites_df

,Launch Site,Lat,Long
0,CCAFS LC-40,28.562302,-80.577356
1,CCAFS SLC-40,28.563197,-80.576820
2,KSC LC-39A,28.573255,-80.646895
3,VAFB SLC-4E,34.632834,-120.610745


Above coordinates are just plain numbers that can not give you any intuitive insights about where are those launch sites. If you are very good at geography, you can interpret those numbers directly in your mind. If not, that's fine too. Let's visualize those locations by pinning them on a map.


We first need to create a folium `Map` object, with an initial center location to be NASA Johnson Space Center at Houston, Texas.


In [7]:
# Start location is NASA Johnson Space Center
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=10)

We could use `folium.Circle` to add a highlighted circle area with a text label on a specific coordinate. For example,


In [8]:
# Create a blue circle at NASA Johnson Space Center's coordinate with a C
circle = folium.Circle(nasa_coordinate, radius=1000, color='#d35400', fill=True).add_child(folium.Popup('NASA Johnson Space Center'))

marker = folium.map.Marker(
    nasa_coordinate,
    # Create an icon as a text label
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'NASA JSC',
        )
    )
site_map.add_child(circle)
site_map.add_child(marker)

and you should find a small yellow circle near the city of Houston and you can zoom-in to see a larger circle.


Now, let's add a circle for each launch site in data frame `launch_sites`


*TODO:*  Create and add `folium.Circle` and `folium.Marker` for each launch site on the site map


An example of folium.Circle:


`folium.Circle(coordinate, radius=1000, color='#000000', fill=True).add_child(folium.Popup(...))`


In [ ]:
# Create a circle with a new color (e.g., green)
    folium.Circle(
        coordinate,
        radius=1000,
        color='#34A853', # A nice green color
        fill=True,
        fill_color='#34A853'
    ).add_child(folium.Popup(site['name'])).add_to(spacex_map)

An example of folium.Marker:


`folium.map.Marker(coordinate, icon=DivIcon(icon_size=(20,20),icon_anchor=(0,0), html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'label', ))`


In [ ]:
 # Create a marker and add it to the marker cluster
    folium.Marker(
        coordinate,
        popup=site['name'],
        icon=folium.Icon(color='blue')
    ).add_to(marker_cluster)

In [9]:
# Start location is NASA Johnson Space Center
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=10)
spacex_map = folium.Map(location=[28.5634, -80.5767], zoom_start=5)

# Initialize a MarkerCluster object
marker_cluster = MarkerCluster().add_to(spacex_map)

# Launch site data
launch_sites = [
    {'name': 'CCAFS LC-40', 'lat': 28.562302, 'long': -80.577356},
    {'name': 'CCAFS SLC-40', 'lat': 28.563197, 'long': -80.576820},
    {'name': 'KSC LC-39A', 'lat': 28.573255, 'long': -80.646895},
    {'name': 'VAFB SLC-4E', 'lat': 34.632834, 'long': -120.610745}
]

# For each launch site, create a marker and a circle
for site in launch_sites:
    coordinate = [site['lat'], site['long']]
    
    # Create a circle with a new color (e.g., green)
    folium.Circle(
        coordinate,
        radius=1000,
        color='#34A853', # A nice green color
        fill=True,
        fill_color='#34A853'
    ).add_child(folium.Popup(site['name'])).add_to(spacex_map)

    # Create a marker and add it to the marker cluster
    folium.Marker(
        coordinate,
        popup=site['name'],
        icon=folium.Icon(color='blue')
    ).add_to(marker_cluster)

# Display the map
spacex_map

Now, you can explore the map by zoom-in/out the marked areas
, and try to answer the following questions:

*   Are all launch sites in proximity to the Equator line?
*   Are all launch sites in very close proximity to the coast?

Also please try to explain your findings.


Next, let's try to enhance the map by adding the launch outcomes for each site, and see which sites have high success rates.
Recall that data frame spacex_df has detailed launch records, and the `class` column indicates if this launch was successful or not


In [8]:
spacex_df.tail(10)

,Launch Site,Lat,Long,class,marker_color
46,KSC LC-39A,28.573255,-80.646895,1,green
47,KSC LC-39A,28.573255,-80.646895,1,green
48,KSC LC-39A,28.573255,-80.646895,1,green
49,CCAFS SLC-40,28.563197,-80.576820,1,green
50,CCAFS SLC-40,28.563197,-80.576820,1,green
51,CCAFS SLC-40,28.563197,-80.576820,0,red
52,CCAFS SLC-40,28.563197,-80.576820,0,red
53,CCAFS SLC-40,28.563197,-80.576820,0,red
54,CCAFS SLC-40,28.563197,-80.576820,1,green
55,CCAFS SLC-40,28.563197,-80.576820,0,red


Next, let's create markers for all launch records.
If a launch was successful `(class=1)`, then we use a green marker and if a launch was failed, we use a red marker `(class=0)`


Note that a launch only happens in one of the four launch sites, which means many launch records will have the exact same coordinate. Marker clusters can be a good way to simplify a map containing many markers having the same coordinate.


Let's first create a `MarkerCluster` object


In [ ]:
marker_cluster = MarkerCluster()

*TODO:* Create a new column in `spacex_df` dataframe called `marker_color` to store the marker colors based on the `class` value


In [ ]:
# Apply a function to check the value of `class` column
# If class=1, marker_color value will be green
# If class=0, marker_color value will be red
spacex_df['marker_color'] = spacex_df['class'].apply(lambda x: 'green' if x == 1 else 'red')

*TODO:* For each launch result in `spacex_df` data frame, add a `folium.Marker` to `marker_cluster`


In [ ]:
marker_cluster = MarkerCluster()
# Create a new column `marker_color`
spacex_df['marker_color'] = spacex_df['class'].apply(lambda x: 'green' if x == 1 else 'red')

## Assuming spacex_df is already loaded and the 
## `marker_color` column is created.
## Assuming a map object `site_map` and a 
## MarkerCluster object `marker_cluster` 
## are already defined.

# Add marker_cluster to current site_map
site_map.add_child(marker_cluster)

## for each row in spacex_df data frame
## create a Marker object with its coordinate
## and customize the Marker's icon property to indicate if this launch was successed or failed, 
# e.g., icon=folium.Icon(color='white', icon_color=row['marker_color']

# For each launch result in spacex_df data frame
for index, record in spacex_df.iterrows():
    ## Create a Marker object with its coordinate and customize its icon
    # TODO: Create and add a Marker cluster to the site map
    marker = folium.Marker(
        location=[record['Lat'], record['Long']],
        popup=f"Launch Site: {record['Launch Site']}<br>Outcome: {'Success' if record['class'] == 1 else 'Failure'}",
        icon=folium.Icon(color=record['marker_color'])
    )  
# Add the marker to the marker_cluster
    marker_cluster.add_child(marker)

# Display the map
site_map

In [7]:
# Task 2: Mark the success/failed launches for each site on the map
import folium
import pandas as pd
from folium.plugins import MarkerCluster

# Assuming spacex_df is already loaded with columns 'Launch Site', 'Lat', 'Long', and 'class'.
# The first step is to create a new column with marker colors
spacex_df['marker_color'] = spacex_df['class'].apply(lambda x: 'green' if x == 1 else 'red')

# Create a single map for all launch locations
# You might want to adjust the zoom start for your specific data
site_map = folium.Map(location=[28.563197, -80.576820], zoom_start=5)

# Initialize a MarkerCluster object for the launch outcomes
marker_cluster = MarkerCluster().add_to(site_map)

# Loop through each record in the DataFrame
for index, record in spacex_df.iterrows():
    # Create a Marker object with its coordinate and color
    marker = folium.Marker(
        location=[record['Lat'], record['Long']],
        popup=f"Launch Site: {record['Launch Site']}<br>Outcome: {'Success' if record['class'] == 1 else 'Failure'}",
        icon=folium.Icon(color=record['marker_color'])
    )
    
    # Add the marker to the marker_cluster
    marker_cluster.add_child(marker)

# Display the map
site_map

In [25]:
# Task1+Task2: Mark all Launch Sites AND: Mark the success/failed launches for each site on the map
# Import necessary libraries
import folium
import pandas as pd
from folium.plugins import MarkerCluster
from js import fetch
import io

# 1. Load the data from the URL
URL = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv'
resp = await fetch(URL)
spacex_csv_file = io.BytesIO((await resp.arrayBuffer()).to_py())
spacex_df = pd.read_csv(spacex_csv_file)

# 2. Select and group the data to get unique launch sites and their coordinates
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()
launch_sites_df = launch_sites_df[['Launch Site', 'Lat', 'Long']]

# 3. Create a map and a MarkerCluster object
# Calculate the average lat/long to center the map
avg_lat = launch_sites_df['Lat'].mean()
avg_long = launch_sites_df['Long'].mean()
site_map = folium.Map(location=[avg_lat, avg_long], zoom_start=4)

# Initialize a MarkerCluster object
marker_cluster = MarkerCluster().add_to(site_map)

# 4. Loop through each row of the launch_sites_df DataFrame
for index, record in launch_sites_df.iterrows():
    # Get the coordinates and launch site name
    coordinate = [record['Lat'], record['Long']]
    site_name = record['Launch Site']

    # Create and add a Circle for each launch site
    folium.Circle(
        coordinate,
        radius=1000,
        color='#34A853',
        fill=True,
        fill_color='#34A853'
    ).add_child(folium.Popup(site_name)).add_to(site_map)

    # Create and add a Marker to the MarkerCluster
    folium.Marker(
        coordinate,
        popup=site_name,
        icon=folium.Icon(color='blue')
    ).add_to(marker_cluster)

# Assuming spacex_df is already loaded with columns 'Launch Site', 'Lat', 'Long', and 'class'.
# The first step is to create a new column with marker colors
spacex_df['marker_color'] = spacex_df['class'].apply(lambda x: 'green' if x == 1 else 'red')

# Create a single map for all launch locations
# You might want to adjust the zoom start for your specific data
site_map = folium.Map(location=[28.563197, -80.576820], zoom_start=5)

# Initialize a MarkerCluster object for the launch outcomes
marker_cluster = MarkerCluster().add_to(site_map)

# Loop through each record in the DataFrame
for index, record in spacex_df.iterrows():
    # Create a Marker object with its coordinate and color
    marker = folium.Marker(
        location=[record['Lat'], record['Long']],
        popup=f"Launch Site: {record['Launch Site']}<br>Outcome: {'Success' if record['class'] == 1 else 'Failure'}",
        icon=folium.Icon(color=record['marker_color'])
    )
    
    # Add the marker to the marker_cluster
    marker_cluster.add_child(marker)

# Display the map
site_map

In [26]:
# Import necessary libraries
import folium
import pandas as pd
from folium.plugins import MarkerCluster
from js import fetch
import io

# 1. Load the data from the URL
URL = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv'
resp = await fetch(URL)
spacex_csv_file = io.BytesIO((await resp.arrayBuffer()).to_py())
spacex_df = pd.read_csv(spacex_csv_file)

# 2. Select and group the data to get unique launch sites and their coordinates for the circles
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()
launch_sites_df = launch_sites_df[['Launch Site', 'Lat', 'Long']]

# 3. Create a single map object to hold all visual elements
site_map = folium.Map(location=[28.563197, -80.576820], zoom_start=5)

# 4. Loop through the unique launch sites to add the green circles
for index, record in launch_sites_df.iterrows():
    # Get the coordinates and launch site name
    coordinate = [record['Lat'], record['Long']]
    site_name = record['Launch Site']

    # Create and add a Circle for each launch site
    folium.Circle(
        coordinate,
        radius=1000,
        color='#34A853',
        fill=True,
        fill_color='#34A853'
    ).add_child(folium.Popup(site_name)).add_to(site_map)

# 5. Create a new column with marker colors based on launch success/failure
spacex_df['marker_color'] = spacex_df['class'].apply(lambda x: 'green' if x == 1 else 'red')

# 6. Initialize a MarkerCluster object and add it to the single map
marker_cluster = MarkerCluster().add_to(site_map)

# 7. Loop through each launch record in the DataFrame to add the success/failure markers to the cluster
for index, record in spacex_df.iterrows():
    # Create a Marker object with its coordinate and color
    marker = folium.Marker(
        location=[record['Lat'], record['Long']],
        popup=f"Launch Site: {record['Launch Site']}<br>Outcome: {'Success' if record['class'] == 1 else 'Failure'}",
        icon=folium.Icon(color=record['marker_color'])
    )
    
    # Add the marker to the marker_cluster
    marker_cluster.add_child(marker)

# 8. Display the final map containing both the circles and the clustered markers
site_map


Your updated map may look like the following screenshots:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_cluster.png">
</center>


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_cluster_zoomed.png">
</center>


From the color-labeled markers in marker clusters, you should be able to easily identify which launch sites have relatively high success rates.


In [9]:
# TASK 3a: Calculate the distances 
# between a launch site to its proximities

# This cell's sole purpose is to provide you with 
# the coordinates tool on the map. Run it, then 
# inspect the map to find and note the coordinates 
# of your target location.

# Assuming 'site_map' from Task 1 is still active
from folium.plugins import MousePosition

# Add Mouse Position to get the coordinate (Lat, Long) for a mouse over on the map
formatter = "function(num) {return L.Util.formatNum(num, 5);};"
mouse_position = MousePosition(
    position='topright',
    separator=' Long: ',
    empty_string='NaN',
    lng_first=False,
    num_digits=20,
    prefix='Lat:',
    lat_formatter=formatter,
    lng_formatter=formatter,
)
site_map.add_child(mouse_position)

# Display the map with the coordinate tool
site_map

In [20]:
# TASK 3b: (ONE LINE) Define Launch Site and Coordinates 
# 1. Define the launch sites and their coordinates
launch_sites = {
    'CCAFS LC-40': {'lat': 28.562302, 'long': -80.577356},
    'CCAFS SLC-40': {'lat': 28.563197, 'long': -80.576820},
    'KSC LC-39A': {'lat': 28.573255, 'long': -80.646895},
    'VAFB SLC-4E': {'lat': 34.632834, 'long': -120.610745}
}

# 2. **MANUALLY EDIT** these two variables for your chosen measurement
selected_site_name = 'VAFB SLC-4E'  # <--- REPLACE with your chosen launch site name
dest_lat = 34.6375      # <--- REPLACE with your manually found latitude
dest_long = -120.624    # <--- REPLACE with your manually found longitude

# ----------------- DO NOT EDIT THE CODE BELOW THIS LINE -----------------

from math import sin, cos, sqrt, atan2, radians
import folium

def calculate_distance(lat1, lon1, lat2, lon2):
    R = 6373.0
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    distance = R * c
    return distance

selected_site = launch_sites[selected_site_name]
start_coord = [selected_site['lat'], selected_site['long']]
end_coord = [dest_lat, dest_long]

calculated_distance = calculate_distance(start_coord[0], start_coord[1], end_coord[0], end_coord[1])

site_map = folium.Map(location=start_coord, zoom_start=12)

lines = folium.PolyLine(locations=[start_coord, end_coord], weight=2, color='blue')
site_map.add_child(lines)

distance_marker = folium.Marker(
    end_coord,
    icon=folium.DivIcon(
        icon_size=(100, 20),
        icon_anchor=(0, 0),
        html=f'<div style="font-size: 12pt; color:#000000;"><b>{calculated_distance:.2f} KM</b></div>'
    )
)
site_map.add_child(distance_marker)

folium.Marker(
    start_coord,
    popup=selected_site_name
).add_to(site_map)

site_map

In [21]:
# TASK 3b: (Multi LINE) Define Launch Site and Coordinates 
# Assuming you have already run the code to create the 'site_map' object
# This code block can be run multiple times to add new lines and markers
# Avoids: site_map = folium.Map(...) as found in version ONE

# Each time you run this second cell with new 
# coordinates, it will add a new folium.PolyLine 
# and a new folium.Marker to the same site_map object

# 1. Manually edit these variables for your new measurement
selected_site_name = 'VAFB SLC-4E'
dest_lat = 34.0522     # New latitude
dest_long = -118.2437  # New longitude

# ----------------- DO NOT EDIT THE CODE BELOW THIS LINE -----------------

# Use the calculate_distance function
start_coord = [launch_sites[selected_site_name]['lat'], launch_sites[selected_site_name]['long']]
end_coord = [dest_lat, dest_long]
calculated_distance = calculate_distance(start_coord[0], start_coord[1], end_coord[0], end_coord[1])

# Create a `folium.PolyLine` object and add it to the existing map
lines = folium.PolyLine(locations=[start_coord, end_coord], weight=2, color='blue')
site_map.add_child(lines)

# Create a marker with a DivIcon and add it to the existing map
distance_marker = folium.Marker(
    end_coord,
    icon=folium.DivIcon(
        icon_size=(100, 20),
        icon_anchor=(0, 0),
        html=f'<div style="font-size: 12pt; color:#000000;"><b>{calculated_distance:.2f} KM</b></div>'
    )
)
site_map.add_child(distance_marker)

# Display the map to see the new line and marker
site_map

In [30]:
# Task 3c: Multi-Line via CSV file data
# Import necessary libraries
import folium
import pandas as pd
from folium.plugins import MarkerCluster
from js import fetch
import io
from math import sin, cos, sqrt, atan2, radians

# Function to calculate distance between two points
def calculate_distance(lat1, lon1, lat2, lon2):
    R = 6373.0  # approximate radius of earth in km
    lat1_rad = radians(lat1)
    lon1_rad = radians(lon1)
    lat2_rad = radians(lat2)
    lon2_rad = radians(lon2)
    dlon = lon2_rad - lon1_rad
    dlat = lat2_rad - lat1_rad
    a = sin(dlat / 2)**2 + cos(lat1_rad) * cos(lat2_rad) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    distance = R * c
    return distance

# 1. Load the main data
URL = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv'
resp = await fetch(URL)
spacex_csv_file = io.BytesIO((await resp.arrayBuffer()).to_py())
spacex_df = pd.read_csv(spacex_csv_file)

# 2. Select and group the data to get unique launch sites and their coordinates
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()
launch_sites_df = launch_sites_df[['Launch Site', 'Lat', 'Long']]

# 3. Create a single map object
site_map = folium.Map(location=[28.563197, -80.576820], zoom_start=5)

# 4. Add circles and markers for launch sites
for index, record in launch_sites_df.iterrows():
    coordinate = [record['Lat'], record['Long']]
    site_name = record['Launch Site']
    folium.Circle(
        coordinate,
        radius=1000,
        color='#34A853',
        fill=True,
        fill_color='#34A853'
    ).add_child(folium.Popup(site_name)).add_to(site_map)
    folium.Marker(
        coordinate,
        popup=site_name,
        icon=folium.Icon(color='blue')
    ).add_to(site_map)

# 5. Load the destinations CSV and draw specific lines
# NOTE: Renamed your CSV file for clarity in this example
URL_dest = 'https://gist.githubusercontent.com/username/your_gist_id/raw/your_file.csv' # Replace with the URL to your CSV
# For this example, we'll load it from a string like you did before.

try:
    # Use your provided CSV data
    csv_data = """Launch_Site_Name,Destination_Name,Destination_Lat,Destination_Long,Class
CCAFS LC-40,Orlando,28.5383,-81.3792,City
CCAFS LC-40,road,28.5814,-80.8522,Road
CCAFS LC-40,rail,28.5719,-80.8039,Rail
KSC LC-39A,coast,28.6129,-80.5987,Coast
KSC LC-39A,road,28.5814,-80.8522,Road
KSC LC-39A,rail,28.5719,-80.8039,Rail
VAFB SLC-4E,Los Angles,34.0452,-118.2437,City
VAFB SLC-4E,coast,34.6375,-120.6241,Coast
VAFB SLC-4E,road,34.6325,-120.6234,Road
VAFB SLC-4E,rail,34.6348,-120.6244,Rail"""
    destinations_df = pd.read_csv(io.StringIO(csv_data))

    # Create a dictionary for line styles based on the 'Class' column
    line_styles = {
        'City': {'color': 'purple', 'weight': 3},
        'Coast': {'color': 'blue', 'weight': 2},
        'Road': {'color': 'green', 'weight': 1.5},
        'Rail': {'color': 'black', 'weight': 1.5}
    }

    # Iterate through your new destinations CSV
    for _, destination_row in destinations_df.iterrows():
        # Get the launch site name from the current row
        launch_site_name = destination_row['Launch_Site_Name']
        
        # Find the coordinates for that specific launch site from our original spacex_df
        # Using .iloc[0] because we expect only one match
        launch_site_coords = launch_sites_df[launch_sites_df['Launch Site'] == launch_site_name]
        
        # Proceed only if we found a matching launch site
        if not launch_site_coords.empty:
            launch_lat = launch_site_coords['Lat'].iloc[0]
            launch_lon = launch_site_coords['Long'].iloc[0]

            # Get destination details from the current row
            dest_lat = destination_row['Destination_Lat']
            dest_lon = destination_row['Destination_Long']
            dest_name = destination_row['Destination_Name']
            dest_class = destination_row['Class']
            
            # Get the style for this line from our dictionary
            style = line_styles.get(dest_class, {'color': 'gray', 'weight': 1}) # Default style
            
            # Define the points for the line
            points = [[launch_lat, launch_lon], [dest_lat, dest_lon]]
            
            # Add the styled PolyLine to the map
            folium.PolyLine(
                points,
                color=style['color'],
                weight=style['weight'],
                opacity=0.8,
                popup=f"From: {launch_site_name}<br>To: {dest_name}"
            ).add_to(site_map)
            
            # Optional: Add a marker for the destination
            distance = calculate_distance(launch_lat, launch_lon, dest_lat, dest_lon)
            folium.Marker(
                [dest_lat, dest_lon],
                popup=f"{dest_name}<br>Distance: {distance:.2f} km",
                icon=folium.Icon(color='red', icon='info-sign')
            ).add_to(site_map)

except Exception as e:
    print(f"An error occurred: {e}")

# Display the map
site_map

Next, we need to explore and analyze the proximities of launch sites.


Let's first add a `MousePosition` on the map to get coordinate for a mouse over a point on the map. As such, while you are exploring the map, you can easily find the coordinates of any points of interests (such as railway)


In [ ]:
# Add Mouse Position to get the coordinate (Lat, Long) for a mouse over on the map
formatter = "function(num) {return L.Util.formatNum(num, 5);};"
mouse_position = MousePosition(
    position='topright',
    separator=' Long: ',
    empty_string='NaN',
    lng_first=False,
    num_digits=20,
    prefix='Lat:',
    lat_formatter=formatter,
    lng_formatter=formatter,
)

site_map.add_child(mouse_position)
site_map

Now zoom in to a launch site and explore its proximity to see if you can easily find any railway, highway, coastline, etc. Move your mouse to these points and mark down their coordinates (shown on the top-left) in order to the distance to the launch site.


Now zoom in to a launch site and explore its proximity to see if you can easily find any railway, highway, coastline, etc. Move your mouse to these points and mark down their coordinates (shown on the top-left) in order to the distance to the launch site.


In [10]:
from math import sin, cos, sqrt, atan2, radians

def calculate_distance(lat1, lon1, lat2, lon2):
    # approximate radius of earth in km
    R = 6373.0

    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    distance = R * c
    return distance

*TODO:* Mark down a point on the closest coastline using MousePosition and calculate the distance between the coastline point and the launch site.


In [12]:
calculate_distance(34.632834, -120.610745, 34.532, -120.627)

11.314101489657183

*TODO:* Draw a `PolyLine` between a launch site to the selected coastline point


In [24]:
# Create and add a folium.Marker on your selected closest coastline point on the map
# Display the distance between coastline point and launch site using the icon property 
# for example
# distance_marker = folium.Marker(
#    coordinate,
#    icon=DivIcon(
#        icon_size=(20,20),
#        icon_anchor=(0,0),
#        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % "{:10.2f} KM".format(distance),
#        )
#    )

# Launch site coordinate
launch_site_coord = [28.5223, -80.57736]

# Closest coastline coordinate (example)
coastline_coord = [28.5637, -80.5678]

distance_coastline = calculate_distance(launch_site_coord[0], launch_site_coord[1], coastline_coord[0], coastline_coord[1])

# Create a folium.PolyLine object to connect the points
line_locations = [launch_site_coord, coastline_coord]
lines = folium.PolyLine(locations=line_locations, weight=2)
site_map.add_child(lines)

# Create a marker at the coastline point to display the distance
distance_marker = folium.Marker(
    coastline_coord,
    icon=folium.DivIcon(
        icon_size=(20, 20),
        icon_anchor=(0, 0),
        html=f'<div style="font-size: 12; color:#000000;"><b>{distance_coastline:.2f} KM</b></div>'
    )
)
site_map.add_child(distance_marker)
site_map

Your updated map with distance line should look like the following screenshot:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_distance.png">
</center>


*TODO:* Similarly, you can draw a line betwee a launch site to its closest city, railway, highway, etc. You need to use `MousePosition` to find the their coordinates on the map first


A railway map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/railway.png">
</center>


A highway map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/highway.png">
</center>


A city map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/city.png">
</center>


After you plot distance lines to the proximities, you can answer the following questions easily:

*   Are launch sites in close proximity to railways?
*   Are launch sites in close proximity to highways?
*   Are launch sites in close proximity to coastline?
*   Do launch sites keep certain distance away from cities?

Also please try to explain your findings.


# Next Steps:

Now you have discovered many interesting insights related to the launch sites' location using folium, in a very interactive way. Next, you will need to build a dashboard using Ploty Dash on detailed launch records.


## Authors


[Pratiksha Verma](https://www.linkedin.com/in/pratiksha-verma-6487561b1/)


<!--## Change Log--!>


<!--| Date (YYYY-MM-DD) | Version | Changed By      | Change Description      |
| ----------------- | ------- | -------------   | ----------------------- |
| 2022-11-09        | 1.0     | Pratiksha Verma | Converted initial version to Jupyterlite|--!>


### <h3 align="center"> IBM Corporation 2022. All rights reserved. <h3/>
